# Notebook 01 — Data Inspection
**CIS6035 IHMS — Somerset Mirissa Beach Hotel**

Goals:
- Load and understand the dataset structure
- Identify data types, missing values, and basic statistics
- Visualise occupancy and revenue time series
- Explore seasonal patterns via boxplots
- Run seasonal decomposition
- Document findings → `docs/data_analysis.md`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

print('Libraries loaded successfully')

In [ ]:
# Load dataset
df = pd.read_csv('../data/hotel_data.csv', parse_dates=['date'], dayfirst=False)
df = df.sort_values('date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
df.head()

In [ ]:
# Data types and nulls
print('=== dtypes ===')
print(df.dtypes)
print('\n=== Null counts ===')
print(df.isnull().sum())

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Value counts for categorical columns
print('=== season value_counts ===')
print(df['season'].value_counts())
print('\n=== is_holiday ===')
print(df['is_holiday'].value_counts())
print('\n=== is_weekend ===')
print(df['is_weekend'].value_counts())

In [ ]:
# Time series: occupancy rate
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(df['date'], df['occupancy_rate'], color='#0ea5e9', alpha=0.8, linewidth=0.8)
axes[0].set_title('Daily Occupancy Rate (2023–2025)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Occupancy Rate')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

axes[1].plot(df['date'], df['revenue'] / 1000, color='#10b981', alpha=0.8, linewidth=0.8)
axes[1].set_title('Daily Revenue (LKR Thousands)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Revenue (000 LKR)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('../docs/figures/01_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
os.makedirs('../docs/figures', exist_ok=True)

# Boxplots by month
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['month_name'] = df['date'].dt.strftime('%b')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

sns.boxplot(data=df, x='month_name', y='occupancy_rate', order=month_order, ax=axes[0], color='#7dd3fc')
axes[0].set_title('Occupancy by Month')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='season', y='occupancy_rate', ax=axes[1],
            order=['very_high','high','medium','low'], color='#a78bfa')
axes[1].set_title('Occupancy by Season')
axes[1].set_xlabel('')

sns.boxplot(data=df, x='is_weekend', y='occupancy_rate', ax=axes[2], color='#6ee7b7')
axes[2].set_title('Occupancy: Weekday vs Weekend')
axes[2].set_xticklabels(['Weekday', 'Weekend'])
axes[2].set_xlabel('')

plt.tight_layout()
plt.savefig('../docs/figures/01_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['occupancy_rate', 'booked_rooms', 'revenue', 'month', 'is_weekend', 'is_holiday']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/figures/01_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Seasonal decomposition (weekly period)
ts = df.set_index('date')['occupancy_rate']
decomp_7 = seasonal_decompose(ts, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(16, 10))
for ax, (name, component) in zip(axes, [
    ('Observed', decomp_7.observed),
    ('Trend', decomp_7.trend),
    ('Seasonal (weekly)', decomp_7.seasonal),
    ('Residual', decomp_7.resid),
]):
    ax.plot(component, linewidth=0.8)
    ax.set_title(name)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('Seasonal Decomposition (period=7)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/figures/01_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key observations summary
print('=== KEY OBSERVATIONS ===')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()} ({len(df)} days)')
print(f'Occupancy range: {df["occupancy_rate"].min():.2f} – {df["occupancy_rate"].max():.2f}')
print(f'Occupancy mean: {df["occupancy_rate"].mean():.3f} ({df["occupancy_rate"].mean()*100:.1f}%)')
print(f'Revenue range: LKR {df["revenue"].min():,.0f} – {df["revenue"].max():,.0f}')
print(f'Season distribution:')
for s, count in df['season'].value_counts().items():
    print(f'  {s}: {count} days ({count/len(df)*100:.1f}%)')
print(f'Holiday days: {df["is_holiday"].sum()}')
print(f'Weekend days: {df["is_weekend"].sum()}')